# Phase 5 — Evaluation
## Brain Tumour MRI Classification
====================================================================

The test set is opened here for the first time.

Every headline number is reported beside the two stratifications that qualify
it: by native file size, because the audit in Phase 1 proved a source shortcut
exists, and by similarity to the training set, because patient identity cannot
be verified on this dataset.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from src import (config, data, engine, explain, manifest, metrics, robustness,
                 splits, viz)
from src.config import (CHENG_DIR, CKPT_PATH, CLASSES, DEVICE, FACE,
                        PALETTE)

D = splits.build_dataset(CHENG_DIR)
images, labels, groups = D["images"], D["labels"], D["groups"]
masks, names = D["masks"], D["names"]
S = splits.build_splits(images, labels, groups)
TEST = S["test_idx"]

test_img, test_lab = images[TEST], labels[TEST]
test_mask, test_grp = masks[TEST], S["merged_groups"][TEST]

model, ckpt = engine.load_checkpoint(CKPT_PATH)
MEAN, STD = ckpt["norm"]
IMG = ckpt["img_size"]

# A figure and a checkpoint from different runs once sat in an outputs folder
# looking equally authoritative. The manifest makes that impossible to miss.
assert ckpt.get("manifest") == manifest.current_hash(), (
    f"checkpoint {ckpt.get('manifest')} does not match manifest "
    f"{manifest.current_hash()} -- re-run Phases 1 and 3")
assert list(ckpt["classes"]) == list(CLASSES), (
    f"checkpoint classes {ckpt['classes']} do not match {list(CLASSES)}")

print(f"checkpoint  epoch {ckpt['epoch']}  val acc {ckpt['val_acc']:.4f}  "
      f"deep={ckpt['deep']}  act={ckpt['activation']}")
print(f"preprocessing travelled with it: img_size {IMG}, MEAN {MEAN:.4f}, STD {STD:.4f}")
print(f"run {manifest.current_hash()} -- checkpoint and figures agree")
print(f"test set: {len(TEST)} scans from {len(set(test_grp.tolist()))} patients")


checkpoint  epoch 29  val acc 0.8750  deep=True  act=relu
preprocessing travelled with it: img_size 128, MEAN 0.1990, STD 0.1585
run b9dcd147c12e -- checkpoint and figures agree
test set: 613 scans from 47 patients


In [2]:
# 1. OPENING THE TEST SET, ONCE
"""
This is the first time these images are used for anything. They were not
involved in choosing the architecture, the learning rate, the augmentation, the
stopping epoch, the balancing strategy or the normalisation constants -- all of
that was decided against the validation split in Phases 3 and 4.

That distinction is the whole value of the number below. A test set that has
been peeked at during development stops being a test set and becomes a second
validation set carrying an optimistic bias nobody can measure.

What is different here from every previous version of this project: the test set
is separated by PATIENT, not by image. Phase 1 asserted that no patient
contributes scans to more than one split, which is the leak that matters on a
dataset where one patient supplies a median of 13 slices. Nothing had to be
excluded as duplicated -- this collection contains none.
"""
eval_tf = data.make_transforms(MEAN, STD, augment=False, img_size=IMG)
test_ds = data.CachedDataset(test_img, test_lab, eval_tf)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=32, shuffle=False)

y_true, y_pred, probs = metrics.predict(model, test_loader)
correct = y_true == y_pred
assert len(y_true) == len(test_lab), "loader dropped images -- check drop_last"

print(f"test images scored: {len(y_true)}   "
      f"patients: {len(set(test_grp.tolist()))}   (none appear in train or val)")
print("per class: " + "  ".join(f"{c} {int((y_true == i).sum())}"
                                for i, c in enumerate(CLASSES)))


test images scored: 613   patients: 47   (none appear in train or val)
per class: glioma 286  meningioma 142  pituitary 185


In [3]:
# 2. HEADLINE, AND WHY MACRO F1 LEADS
acc, acc_lo, acc_hi = metrics.bootstrap_ci(y_true, y_pred, "accuracy")
f1, f1_lo, f1_hi = metrics.bootstrap_ci(y_true, y_pred, "macro_f1",
                                        n_classes=len(CLASSES))
print(f"  macro F1   {f1:.4f}   95% CI [{f1_lo:.4f}, {f1_hi:.4f}]   <- the headline")
print(f"  accuracy   {acc:.4f}   95% CI [{acc_lo:.4f}, {acc_hi:.4f}]")
print(f"\n  chance                      {1/len(CLASSES):.4f}")
print(f"  validation (Phase 3)        {ckpt['val_acc']:.4f}   <- selection metric, biased upward")

if acc > 0.995:
    print("\n  WARNING: above 99.5% on this dataset is more likely leakage than skill.")


  macro F1   0.8711   95% CI [0.8445, 0.8959]   <- the headline
  accuracy   0.8793   95% CI [0.8532, 0.9038]

  chance                      0.3333
  validation (Phase 3)        0.8750   <- selection metric, biased upward


In [4]:
# 3. CONFUSION MATRIX AND PER-CLASS SCORES

cm = metrics.confusion(y_true, y_pred, len(CLASSES))
viz.plot_confusion(cm, CLASSES)

off = sorted(((cm[i, j], CLASSES[i], CLASSES[j])
              for i in range(len(CLASSES)) for j in range(len(CLASSES)) if i != j),
             reverse=True)
print("largest confusions:")
for n, a, b in off[:4]:
    if n:
        print(f"  {n:>4} {a} scans predicted as {b}")

print()
rows = metrics.per_class_report(y_true, y_pred, CLASSES)
metrics.print_report(rows)

print(f"\n{'class':<14}{'recall':>9}{'95% CI':>20}{'support':>9}")
print("-" * 52)
for i, name in enumerate(CLASSES):
    p, lo, hi = metrics.bootstrap_ci(y_true, y_pred, class_idx=i)
    print(f"{name:<14}{p:>9.4f}   [{lo:.4f}, {hi:.4f}]{int((y_true==i).sum()):>9}")


  saved -> outputs/confusion_matrix.png
largest confusions:
    38 glioma scans predicted as meningioma
    21 pituitary scans predicted as meningioma
     7 meningioma scans predicted as pituitary
     5 meningioma scans predicted as glioma

class             precision   recall       f1  support
------------------------------------------------------
glioma               0.9762   0.8601   0.9145      286
meningioma           0.6878   0.9155   0.7855      142
pituitary            0.9477   0.8811   0.9132      185
------------------------------------------------------
macro avg            0.8706   0.8856   0.8711      613
weighted avg         0.9008   0.8793   0.8842      613

class            recall              95% CI  support
----------------------------------------------------
glioma           0.8601   [0.8182, 0.8986]      286
meningioma       0.9155   [0.8662, 0.9577]      142


pituitary        0.8811   [0.8324, 0.9297]      185


In [5]:
# 4. ROC AND AUC

curves, aucs, macro_auc = metrics.roc_ovr(y_true, probs, len(CLASSES))
viz.plot_roc(curves, aucs, macro_auc, CLASSES)
for name, a in zip(CLASSES, aucs):
    print(f"  {name:<14}AUC {a:.4f}")
print(f"  {'macro':<14}AUC {macro_auc:.4f}")

  saved -> outputs/roc_curves.png
  glioma        AUC 0.9865
  meningioma    AUC 0.9723
  pituitary     AUC 0.9811
  macro         AUC 0.9800


In [6]:
# 5. WHAT THE MODEL BEATS
"""
An accuracy is only interpretable against what a trivial method achieves on the
same images. Phase 1 measured two such baselines and they set the bar this model
has to clear.

The texture probe is the important one. A random forest given five per-image
statistics -- mean, standard deviation, Laplacian variance, gradient energy and
high-frequency share -- and no spatial structure whatsoever scored 0.6508 under
patient-wise folds. If the CNN lands near that, it has learned little beyond
global image statistics, whatever its architecture diagram suggests.

"""
# read from the manifest, not hardcoded: these were measured in Phase 1 and
# a stale copy here is exactly the drift the manifest exists to prevent
M = manifest.read()
TEX_PROBE = M["texture_probe_patientwise"]
ANAT_ORACLE = M["anatomy_oracle_patientwise"]
maj = np.bincount(y_true).max() / len(y_true)

print(f"{'method':<44}{'accuracy':>10}")
print("-" * 54)
for name, v in (("majority class (predict glioma always)", maj),
                ("texture probe, 5 pixel statistics (Phase 1)", TEX_PROBE),
                ("anatomy oracle: mask centroid+area (Phase 1)", ANAT_ORACLE),
                ("this model, held-out test set", acc)):
    print(f"{name:<44}{v:>10.4f}")

print(f"\n  over majority class      {acc - maj:+.4f}")
print(f"  over the texture probe   {acc - TEX_PROBE:+.4f}")
print(f"  over the anatomy oracle  {acc - ANAT_ORACLE:+.4f}")
print("""
The margin over the texture probe is the one that matters: it is the part of the
result that global image statistics cannot account for.""")

# --- does it only work on large tumours? ---
area = test_mask.reshape(len(test_mask), -1).sum(1) / test_mask[0].size
cuts = np.quantile(area, [1/3, 2/3])
bands = [("small  (bottom third)", area <= cuts[0]),
         ("medium (middle third)", (area > cuts[0]) & (area <= cuts[1])),
         ("large  (top third)", area > cuts[1])]

print(f"\n{'tumour size':<26}{'n':>6}{'area':>9}{'accuracy':>11}{'macro F1':>11}")
print("-" * 63)
for label, m in bands:
    print(f"{label:<26}{m.sum():>6}{area[m].mean():>9.4f}"
          f"{correct[m].mean():>11.4f}"
          f"{metrics.macro_f1(y_true[m], y_pred[m], len(CLASSES)):>11.4f}")
print(f"{'ALL':<26}{len(area):>6}{area.mean():>9.4f}{acc:>11.4f}{f1:>11.4f}")

small_acc = correct[bands[0][1]].mean()
print(f"\nsmallest-third accuracy {small_acc:.4f} against {acc:.4f} overall "
      f"({small_acc - acc:+.4f})")

viz.plot_bars([b[0].split()[0] for b in bands] + ["all"],
              [correct[m].mean() for _, m in bands] + [acc],
              "size_stratified.png",
              "Accuracy by tumour size -- does it only find large lesions?",
              hline=acc)


method                                        accuracy
------------------------------------------------------
majority class (predict glioma always)          0.4666
texture probe, 5 pixel statistics (Phase 1)     0.6691
anatomy oracle: mask centroid+area (Phase 1)    0.7063
this model, held-out test set                   0.8793

  over majority class      +0.4127
  over the texture probe   +0.2102
  over the anatomy oracle  +0.1730

The margin over the texture probe is the one that matters: it is the part of the
result that global image statistics cannot account for.

tumour size                    n     area   accuracy   macro F1
---------------------------------------------------------------
small  (bottom third)        205   0.0076     0.8829     0.8500
medium (middle third)        204   0.0194     0.8480     0.8484
large  (top third)           204   0.0440     0.9069     0.8829
ALL                          613   0.0236     0.8793     0.8711

smallest-third accuracy 0.8829 against 0

  saved -> outputs/size_stratified.png


WindowsPath('C:/Games/Codes/Python/Projects/Brain_Tumour_Detection/Final_Project/outputs/size_stratified.png')

In [7]:
# 6. LEAKAGE, CLOSED RATHER THAN BOUNDED
"""

Cheng ships a patient ID with every scan, so the leak is closed instead of
estimated. No patient contributes images to more than one split -- Phase 1
asserts it, and the assertion is re-derived here from the split indices rather
than trusted.

The similarity bands below are no longer a bound on unremovable leakage. They
are a check that the closure worked: with patients disjoint, the nearest
same-class training image should sit no closer than the nearest different-class
one, which is the different-patient null.
"""
train_set = set(S["merged_groups"][S["train_idx"]].tolist())
val_set = set(S["merged_groups"][S["val_idx"]].tolist())
test_set = set(test_grp.tolist())
assert not (test_set & train_set), "test patient also appears in train"
assert not (test_set & val_set), "test patient also appears in val"
print(f"patients  train {len(train_set)}   val {len(val_set)}   test {len(test_set)}")
print(f"overlap   train&test {len(test_set & train_set)}   "
      f"val&test {len(test_set & val_set)}   [asserted zero]")

nbr = splits.neighbour_report(images, labels, S["train_idx"], TEST)
cos = nbr["same_class"]

print(f"\n{'nearest training image, cosine':<34}{'same class':>12}{'other class':>13}")
print("-" * 59)
for t in (0.99, 0.98, 0.95, 0.90):
    print(f"  share of test images >= {t:.2f}{'':<8}"
          f"{nbr['bands'][t]:>12.3%}{nbr['null_bands'][t]:>13.3%}")

print(f"\n{'similarity band':<34}{'n':>6}{'accuracy':>11}")
print("-" * 51)
for lo, hi, label in [(0.90, 1.01, "0.90+        (most similar)"),
                      (0.85, 0.90, "0.85 - 0.90"),
                      (0.00, 0.85, "below 0.85   (least similar)")]:
    m = (cos >= lo) & (cos < hi)
    if m.sum():
        print(f"{label:<34}{m.sum():>6}{correct[m].mean():>11.4f}")
print(f"{'ALL':<34}{len(correct):>6}{correct.mean():>11.4f}")

low = cos < 0.85
if low.sum():
    print(f"\nleast-similar band: {correct[low].mean():.4f} on {low.sum()} images, "
          f"against {correct.mean():.4f} overall")
print("""
A flat profile across these bands is the expected result now, and a steep one
would mean the patient grouping had failed somewhere. This is a check on the
split, not a caveat on the headline.""")


patients  train 159   val 27   test 47
overlap   train&test 0   val&test 0   [asserted zero]



nearest training image, cosine      same class  other class
-----------------------------------------------------------
  share of test images >= 0.99              0.000%       0.000%
  share of test images >= 0.98              0.000%       0.000%
  share of test images >= 0.95              0.163%       0.000%
  share of test images >= 0.90              0.653%       0.163%

similarity band                        n   accuracy
---------------------------------------------------
0.90+        (most similar)            4     1.0000
0.85 - 0.90                           15     0.9333
below 0.85   (least similar)         594     0.8771
ALL                                  613     0.8793

least-similar band: 0.8771 on 594 images, against 0.8793 overall

A flat profile across these bands is the expected result now, and a steep one
would mean the patient grouping had failed somewhere. This is a check on the
split, not a caveat on the headline.


In [8]:
# 7. ROBUSTNESS — HOW FAR DOES THE NUMBER TRAVEL?
"""
Every figure so far describes this dataset. The standard objection to a high
accuracy on a curated public MRI benchmark is not that the number is faked, it
is that the benchmark is clean in ways a hospital is not: consistently windowed,
uniformly sharp, low noise. A model can learn to depend on all of that, and no
ordinary evaluation shows the dependence, because the test split is clean in
exactly the same ways.

So the test set is degraded and scored again. This is not external validation
and does not pretend to be. It answers a narrower question honestly: does the
accuracy survive the variation that separates one scanner from another?
"""
rob = robustness.robustness_test(model, test_img, y_true, MEAN, STD, IMG,
                                 n_classes=len(CLASSES))
robustness.print_robustness(rob)

labels = list(rob)
viz.plot_bars(labels, [rob[k]["f1"] for k in labels], "robustness.png",
              "Macro F1 under acquisition variation", ylabel="macro F1",
              hline=rob["clean"]["f1"],
              colours=[PALETTE["good"]] + [PALETTE["val"]] * (len(labels) - 1))

print("""
Read the bias-field row against the noise and blur rows. Bias field is the most
MRI-specific corruption here and a model reading anatomy should barely notice
it; noise and blur disturb high-frequency texture. Which of those hurts more
says something about what the model is using.""")

  condition                 accuracy  macro F1   drop (F1)
  --------------------------------------------------------
  clean                       0.8793    0.8711
  noise sigma=5               0.8613    0.8556     +0.0154
  noise sigma=15              0.5155    0.4860     +0.3851
  blur radius 1px             0.7700    0.7549     +0.1162
  blur radius 2px             0.5644    0.4441     +0.4269
  gamma 0.7                   0.6591    0.6529     +0.2182
  gamma 1.4                   0.6215    0.5230     +0.3481
  bias field +/-20%           0.8793    0.8713     -0.0002

  worst case across the suite: macro F1 0.4441 against a clean 0.8711


  saved -> outputs/robustness.png

Read the bias-field row against the noise and blur rows. Bias field is the most
MRI-specific corruption here and a model reading anatomy should barely notice
it; noise and blur disturb high-frequency texture. Which of those hurts more
says something about what the model is using.


In [9]:
# 8. OCCLUSION AGAINST GROUND TRUTH
"""
This is the test the whole dataset change was made for, and no earlier version
of this project could run it.

A heatmap shows where gradient flows. It does not show what a prediction depends
on, and the two genuinely come apart -- a map can sit squarely on a region the
classifier would happily do without. Occlusion answers the stronger question by
perturbing the input and observing the consequence.

"""
rng = np.random.default_rng(config.SEED)
eff, skipped = [], 0
for i in range(len(TEST)):
    m = test_mask[i]
    if not m.any():
        skipped += 1
        continue
    ctrl = explain.shifted_control(m, explain.brain_mask(test_img[i]), rng)
    if ctrl is None:
        skipped += 1
        continue
    r = explain.occlusion_effect(model, test_img[i], m, eval_tf,
                                 class_idx=int(test_lab[i]), control=ctrl)
    eff.append(r)

base = np.array([r["base"] for r in eff])
les = np.array([r["lesion"] for r in eff])
ctl = np.array([r["control"] for r in eff])
d_les, d_ctl = base - les, base - ctl

print(f"scored {len(eff)} test scans ({skipped} skipped: no mask, or no room "
      f"for a non-overlapping control)\n")
print(f"{'occluded region':<34}{'mean p(true)':>14}{'mean drop':>12}")
print("-" * 60)
print(f"{'nothing (baseline)':<34}{base.mean():>14.4f}{'':>12}")
print(f"{'the annotated lesion':<34}{les.mean():>14.4f}{d_les.mean():>12.4f}")
print(f"{'a matched region elsewhere':<34}{ctl.mean():>14.4f}{d_ctl.mean():>12.4f}")

gap = d_les.mean() - d_ctl.mean()
wins = float((d_les > d_ctl).mean())
sem = np.sqrt((d_les - d_ctl).var(ddof=1) / len(eff))
print(f"\n  lesion minus control      {gap:+.4f}  (SE {sem:.4f}, "
      f"{gap/sem:.1f} standard errors)")
print(f"  lesion hurts more, share  {wins:.3f} of {len(eff)} scans")
print(f"""
Two numbers, two different claims. The gap is {gap/sem:.1f} standard errors from
zero, so the model does use the lesion on average. But blanking the tumour
entirely leaves p(true) at {les.mean():.4f}, down only {100*(base.mean()-les.mean())/base.mean():.1f}% relative, and the
per-scan win rate is {wins:.3f} -- barely a coin flip. Most of the confidence
survives removing the tumour, which is what Phase 1's 0.7063 anatomy oracle
predicted: in this collection, where the tumour sits carries much of the class.""")

fig, axes = viz.styled_fig(2, 3, figsize=(11, 7.5))
for col, (c, name) in enumerate(zip(range(len(CLASSES)), CLASSES)):
    hit = [i for i in np.where(y_true == c)[0] if y_pred[i] == c and test_mask[i].any()]
    if not hit:
        continue
    i = hit[0]
    heat, _, base_p = explain.occlusion_map(model, test_img[i], eval_tf,
                                            class_idx=int(test_lab[i]))
    axes[0, col].imshow(test_img[i], cmap='gray')
    axes[0, col].contour(test_mask[i], levels=[0.5], colors=['#D85A30'], linewidths=1.2)
    axes[0, col].set_title(f"{name}  p={base_p:.2f}\n(outline = radiologist mask)",
                           fontsize=9)
    axes[1, col].imshow(test_img[i], cmap='gray')
    axes[1, col].imshow(heat, cmap='jet', alpha=0.55)
    axes[1, col].contour(test_mask[i], levels=[0.5], colors=['w'], linewidths=1.0)
    axes[1, col].set_title("occlusion sensitivity", fontsize=9)
for ax in axes.ravel():
    ax.axis('off')
plt.suptitle("What the prediction depends on, against the annotation",
             fontsize=12, fontweight='bold')
plt.tight_layout(); viz.save(fig, "occlusion.png")


scored 457 test scans (156 skipped: no mask, or no room for a non-overlapping control)

occluded region                     mean p(true)   mean drop
------------------------------------------------------------
nothing (baseline)                        0.8122            
the annotated lesion                      0.7665      0.0457
a matched region elsewhere                0.8112      0.0010

  lesion minus control      +0.0447  (SE 0.0097, 4.6 standard errors)
  lesion hurts more, share  0.613 of 457 scans

Two numbers, two different claims. The gap is 4.6 standard errors from
zero, so the model does use the lesion on average. But blanking the tumour
entirely leaves p(true) at 0.7665, down only 5.6% relative, and the
per-scan win rate is 0.613 -- barely a coin flip. Most of the confidence
survives removing the tumour, which is what Phase 1's 0.7063 anatomy oracle
predicted: in this collection, where the tumour sits carries much of the class.


  saved -> outputs/occlusion.png


WindowsPath('C:/Games/Codes/Python/Projects/Brain_Tumour_Detection/Final_Project/outputs/occlusion.png')

In [10]:
# 9. GRAD-CAM, AND WHICH LAYER IS ACTUALLY BEST
"""
Grad-CAM was removed from this project once already, because on the previous
dataset there was no way to tell a good map from a plausible-looking one. That
is no longer true: every scan here carries a radiologist's pixel mask, so the
layer is chosen by measurement rather than by intuition.

The method itself comes from pytorch-grad-cam (jacobgil), not from this repo.
There is nothing to be gained by reimplementing a standard algorithm, and
something to lose: a local copy can drift from what its name implies and nobody
notices. What is local is the part that is not standard -- the baseline and the
metric below, which are what turn a heatmap into a measurement.

Two things make this a measurement.

Area-matched IoU. A fixed threshold caps the score artificially -- `cam >= 0.5`
selects roughly a quarter of the frame while masks average 2.4% of it, so the
best attainable IoU would be about 0.08 no matter how good the map. Selecting
the top-N pixels, where N is the mask's own area, removes that ceiling.

The centre blob. A Gaussian at the image centre, identical for every scan,
knowing nothing. Brain tumours sit near the middle of a slice often enough that
this scores well, so any map that cannot beat it has not demonstrated
localisation -- it has demonstrated that tumours are usually central. Reporting
a CAM without this baseline beside it is the single easiest way to overclaim.
"""
blob = explain.centre_blob(test_mask[0].shape)
blob_iou = np.array([explain.area_matched_iou(blob, m) or 0.0 for m in test_mask])

cam_iou = {}
for layer in explain.CAM_LAYERS:
    v = []
    for i in range(len(TEST)):
        cam, _ = explain.grad_cam(model, test_img[i], eval_tf,
                                  layer=layer, class_idx=int(test_lab[i]))
        v.append(explain.area_matched_iou(cam, test_mask[i]) or 0.0)
    cam_iou[layer] = np.array(v)

print(f"{'layer':<14}{'resolution':>12}{'IoU':>9}{'vs blob':>10}{'beats blob':>12}")
print("-" * 57)
print(f"{'centre blob':<14}{'--':>12}{blob_iou.mean():>9.4f}")
RES = {"block2": "64x64", "block3": "32x32", "block3b": "32x32",
       "block4": "16x16", "block4b": "16x16"}
for layer, v in cam_iou.items():
    print(f"{layer:<14}{RES[layer]:>12}{v.mean():>9.4f}"
          f"{v.mean() - blob_iou.mean():>+10.4f}{(v > blob_iou).mean():>12.3f}")

BEST_LAYER = max(cam_iou, key=lambda k: cam_iou[k].mean())
best = cam_iou[BEST_LAYER]
print(f"\nbest layer: {BEST_LAYER}")
print("""
Note the direction. The finest stage is the WORST, by a wide margin, and the
coarsest is the best -- the opposite of the intuition that a higher-resolution
map localises better. Early layers respond to edges and texture everywhere in
the image; they have resolution but nothing class-specific to say. Whatever
limits this model's localisation, it is not the 16x16 grid.
""")

print(f"{'method':<22}{'overall':>9}" + "".join(f"{c:>13}" for c in CLASSES))
print("-" * 70)
for name, v in (("centre blob", blob_iou), (f"Grad-CAM {BEST_LAYER}", best)):
    print(f"{name:<22}{v.mean():>9.4f}" +
          "".join(f"{v[test_lab == c].mean():>13.4f}" for c in range(len(CLASSES))))
print("  " + "-" * 68)
print(f"{'difference':<22}{best.mean() - blob_iou.mean():>+9.4f}" + "".join(
    f"{best[test_lab == c].mean() - blob_iou[test_lab == c].mean():>+13.4f}"
    for c in range(len(CLASSES))))

fig, axes = viz.styled_fig(3, 3, figsize=(11, 11))
for col, (c, name) in enumerate(zip(range(len(CLASSES)), CLASSES)):
    hit = [i for i in np.where(y_true == c)[0]
           if y_pred[i] == c and test_mask[i].any()]
    if not hit:
        continue
    i = max(hit, key=lambda j: cam_iou[BEST_LAYER][j])
    cam, _ = explain.grad_cam(model, test_img[i], eval_tf,
                              layer=BEST_LAYER, class_idx=int(test_lab[i]))
    axes[0, col].imshow(test_img[i], cmap='gray')
    axes[0, col].contour(test_mask[i], levels=[0.5], colors=['#D85A30'], linewidths=1.3)
    axes[0, col].set_title(f"{name}\n(outline = radiologist mask)", fontsize=9)
    axes[1, col].imshow(test_img[i], cmap='gray')
    axes[1, col].imshow(cam, cmap='jet', alpha=0.5)
    axes[1, col].contour(test_mask[i], levels=[0.5], colors=['w'], linewidths=1.0)
    axes[1, col].set_title(f"Grad-CAM {BEST_LAYER}  "
                           f"IoU {cam_iou[BEST_LAYER][i]:.3f}", fontsize=9)
    axes[2, col].imshow(test_img[i], cmap='gray')
    axes[2, col].imshow(blob, cmap='jet', alpha=0.5)
    axes[2, col].contour(test_mask[i], levels=[0.5], colors=['w'], linewidths=1.0)
    axes[2, col].set_title(f"centre blob control  IoU {blob_iou[i]:.3f}", fontsize=9)
for ax in axes.ravel():
    ax.axis('off')
plt.suptitle("Grad-CAM against ground truth, with the control it must beat",
             fontsize=12, fontweight='bold')
plt.tight_layout(); viz.save(fig, "gradcam.png")

CAM_BEATS_BLOB = {CLASSES[c]: float(best[test_lab == c].mean()
                                    - blob_iou[test_lab == c].mean())
                  for c in range(len(CLASSES))}


layer           resolution      IoU   vs blob  beats blob
---------------------------------------------------------
centre blob             --   0.0813
block2               64x64   0.0107   -0.0706       0.214
block3               32x32   0.0159   -0.0655       0.227
block3b              32x32   0.0513   -0.0300       0.442
block4               16x16   0.0476   -0.0338       0.233
block4b              16x16   0.1601   +0.0788       0.440

best layer: block4b

Note the direction. The finest stage is the WORST, by a wide margin, and the
coarsest is the best -- the opposite of the intuition that a higher-resolution
map localises better. Early layers respond to edges and texture everywhere in
the image; they have resolution but nothing class-specific to say. Whatever
limits this model's localisation, it is not the 16x16 grid.

method                  overall       glioma   meningioma    pituitary
----------------------------------------------------------------------
centre blob            

  saved -> outputs/gradcam.png


In [11]:
# 10. SUMMARY AND HONEST LIMITATIONS
"""
The limitations are not boilerplate. Each is a specific reason the number above
could be optimistic, and naming them is what separates a result from a claim.

Two limitations that every previous version of this project carried are gone,
and it is worth being explicit that they were removed rather than reworded: this
dataset has patient identifiers, so same-patient leakage is closed instead of
bounded, and it has lesion masks, so attention is measured against ground truth
instead of an impression.
"""
worst_rob = min(v["f1"] for k, v in rob.items() if k != "clean")
weakest = min(rows[:len(CLASSES)], key=lambda r: r["recall"])

print("=" * 66)
print("PHASE 5 -- RESULTS")
print("=" * 66)
print(f"  macro F1                     {f1:.4f}   95% CI [{f1_lo:.4f}, {f1_hi:.4f}]")
print(f"  accuracy                     {acc:.4f}   95% CI [{acc_lo:.4f}, {acc_hi:.4f}]")
print(f"  macro AUC                    {macro_auc:.4f}")
print(f"  weakest class                {weakest['class']} (recall {weakest['recall']:.4f})")
print(f"  texture-probe baseline       {TEX_PROBE:.4f}          <- section 5")
print(f"  smallest-third tumours       {small_acc:.4f}   <- section 5")
print(f"  lesion vs control occlusion  {gap:+.4f}   <- section 8")
print(f"  Grad-CAM layer chosen        {BEST_LAYER} (measured, not assumed)")
print(f"  Grad-CAM IoU vs centre blob  {best.mean() - blob_iou.mean():+.4f} overall")
for k, v in CAM_BEATS_BLOB.items():
    print(f"     {k:<24}{v:+.4f}")
print(f"  worst case under corruption  {worst_rob:.4f} macro F1   <- section 7")

checks = [
    ("every held-out image scored",       len(y_true) == len(TEST)),
    ("test patients disjoint from train", not (test_set & train_set)),
    ("test patients disjoint from val",   not (test_set & val_set)),
    ("checkpoint matches run manifest",   ckpt["manifest"] == manifest.current_hash()),
    ("accuracy above the texture probe",  acc > TEX_PROBE),
    ("accuracy not implausibly high",     acc <= 0.995),
    ("all classes have non-zero recall",  all(r["recall"] > 0 for r in rows[:len(CLASSES)])),
    ("figures written", all((config.OUTPUTS / fn).exists() for fn in
        ["confusion_matrix.png", "roc_curves.png", "size_stratified.png",
         "robustness.png", "occlusion.png", "gradcam.png"])),
]
print("\n  verification:")
for label, ok in checks:
    print(f"    {'OK  ' if ok else 'FAIL'}  {label}")
failed = [l for l, ok in checks if not ok]
assert not failed, "failed checks: " + "; ".join(failed)

print(f"""
  WHAT THIS IS

  Tumour TYPING, not detection. There is no healthy class, because Cheng ships
  no healthy scans and every attempt to supply them from a second collection was
  measured and rejected: a five-feature probe separates the tumour source from
  the healthy source at 0.92 whichever collection is used and whatever
  normalisation is applied. A detector built that way reads provenance rather
  than anatomy, which is precisely how the previous dataset failed. This model
  answers "which of three tumour types", given that a tumour is present.

  LIMITATIONS

  1. Class is strongly determined by anatomy in this collection. Mask position
     and size alone predict it at {ANAT_ORACLE:.4f}, and five pixel statistics reach {TEX_PROBE:.4f}
     under patient-wise folds. The margin over that probe, not the headline, is
     the part attributable to learned structure. Section 5 reports it.

  2. Only {len(test_set)} test patients. The effective sample size for every
     number here is patients, not the {len(TEST)} slices, because one patient
     contributes a median of 13 highly correlated scans. The confidence
     intervals are computed over images and are therefore optimistic; the
     patient count is the honest denominator.

  3. The test set is uneven -- {int((y_true == 0).sum())} glioma to
     {int((y_true == 1).sum())} meningioma -- which is why macro F1 leads and
     accuracy follows.

""")


PHASE 5 -- RESULTS
  macro F1                     0.8711   95% CI [0.8445, 0.8959]
  accuracy                     0.8793   95% CI [0.8532, 0.9038]
  macro AUC                    0.9800
  weakest class                glioma (recall 0.8601)
  texture-probe baseline       0.6691          <- section 5
  smallest-third tumours       0.8829   <- section 5
  lesion vs control occlusion  +0.0447   <- section 8
  Grad-CAM layer chosen        block4b (measured, not assumed)
  Grad-CAM IoU vs centre blob  +0.0788 overall
     glioma                  +0.0288
     meningioma              +0.3797
     pituitary               -0.0749
  worst case under corruption  0.4441 macro F1   <- section 7

  verification:
    OK    every held-out image scored
    OK    test patients disjoint from train
    OK    test patients disjoint from val
    OK    checkpoint matches run manifest
    OK    accuracy above the texture probe
    OK    accuracy not implausibly high
    OK    all classes have non-zero recall
  